<a href="https://colab.research.google.com/github/Sheulynath/Assignment/blob/Request/new2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Shell
pip install torch torchvision typing-extensions>=4.11


In [ ]:
# PyTorch and training dependencies
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import os


# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define data paths
train_dir = '/content/drive/MyDrive/AD_dataset/copy/TrainAD'
test_dir = '/content/drive/MyDrive/AD_dataset/copy/TestAD'

# Data augmentation and normalization
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), shear=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
test_data = datasets.ImageFolder(test_dir, transform=val_test_transforms)

# Split train into train/val
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(train_data, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# Load pre-trained model
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(128, 1),
    nn.Sigmoid()
)
model = model.to(device)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]


In [ ]:
# Define loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).cpu().numpy()
        preds = (outputs > 0.5).astype(int).flatten()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

# Metrics
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=test_data.classes))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Epoch [1/10], Loss: 0.6921
Epoch [2/10], Loss: 0.6398
Epoch [3/10], Loss: 0.6084
Epoch [4/10], Loss: 0.6077
Epoch [5/10], Loss: 0.6189
Epoch [6/10], Loss: 0.6018
Epoch [7/10], Loss: 0.5776
Epoch [8/10], Loss: 0.5581
Epoch [9/10], Loss: 0.5671
Epoch [10/10], Loss: 0.5223
Classification Report:
              precision    recall  f1-score   support

          AD       0.76      0.58      0.66       270
          HP       0.61      0.79      0.69       231

    accuracy                           0.67       501
   macro avg       0.69      0.68      0.67       501
weighted avg       0.69      0.67      0.67       501

Confusion Matrix:
[[156 114]
 [ 49 182]]


In [ ]:

# Save PyTorch model
torch.save(model.state_dict(), "graphad_model.pth")

In [ ]:
# ===============================
# 📦 Import Required Libraries
# ===============================
!pip install onnx
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import os

# ===============================
# ⚙️ Setup Device
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# 📁 Load Your Trained Model
# (Ensure model is trained or loaded here)
# ===============================
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(128, 1),
    nn.Sigmoid()
)
# Load saved model state_dict (optional)
# model.load_state_dict(torch.load("graphad_model.pth"))
model.eval()
model.to(device)

# ===============================
# 🔄 Convert to ONNX Format
# ===============================
dummy_input = torch.randn(1, 3, 224, 224).to(device)  # (batch_size, channels, height, width)
onnx_model_path = "graphad_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_model_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=11
)

print(f"✅ Model exported to ONNX format: {onnx_model_path}")

# ===============================
# 📤 (Optional) Download in Colab
# ===============================
from google.colab import files
files.download(onnx_model_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 57.1 MB/s eta 0:00:00
✅ Model exported to ONNX format: graphad_model.onnx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>